# Проект «Анализ вакансий»
   

In [26]:
import pandas as pd
import psycopg2

In [3]:
# вставьте сюда параметры подключения из юнита 1. Работа с базой данных из Python

In [28]:
DBNAME = 'project_sql'
USER = 'skillfactory'
PASSWORD = 'cCkxxLVrDE8EbvjueeMedPKt'
HOST = '84.201.134.129'
PORT = 5432

In [74]:
connection = psycopg2.connect(
    dbname=DBNAME,
    user=USER,
    host=HOST,
    password=PASSWORD,
    port=PORT
)

In [75]:
# Создание курсора
cur = connection.cursor()


In [76]:
cur.execute("SELECT to_regclass('vacancies');")
print(cur.fetchone())  # (None,) если таблицы нет

('vacancies',)


## 3. Предварительный анализ данных

1. Напишите запрос, который посчитает количество вакансий в базе (вакансии находятся в таблице `vacancies`).

In [80]:
query_3_1 = """
SELECT 
    COUNT(vacancies.*) AS vacancies_count
FROM 
     public.vacancies 
"""

In [81]:
cur.execute(query_3_1)

In [82]:
# Получение результатов
result_1 = cur.fetchone()  # fetchone() для одной строки результата

# Вывод результата
print(f"Общее количество вакансий: {result_1[0]}")

Общее количество вакансий: 49197


In [73]:
cur.close()

2. Напишите запрос, который посчитает количество работодателей (таблица `employers`).

In [83]:
# текст запроса
query_3_2 = """
SELECT 
    COUNT(employers.*) AS employers_count
FROM 
     public.employers
"""

In [84]:
# результат запроса
cur.execute(query_3_2)

In [85]:
# Получение результатов
result_2 = cur.fetchone()  # fetchone() для одной строки результата

# Вывод результата
print(f"Общее количество работодателей: {result_2[0]}")

Общее количество работодателей: 23501


3. Посчитайте с помощью запроса количество регионов (таблица `areas`).

In [86]:
# текст запроса
query_3_3 = """
SELECT 
    COUNT(areas.*) AS areas_count
FROM 
     public.areas
"""

In [87]:
# результат запроса
cur.execute(query_3_3)

result_3 = cur.fetchone() 

# Вывод результата
print(f"Общее количество вакансий: {result_3[0]}")

Общее количество вакансий: 1362


4. Посчитайте с помощью запроса количество сфер деятельности в базе (таблица `industries`).

In [88]:
# текст запроса
query_3_4 = """
SELECT 
    COUNT(industries.*) AS industries_count
FROM 
     public.industries
"""


In [89]:
# результат запроса
cur.execute(query_3_4)

result_4 = cur.fetchone() 

# Вывод результата
print(f"Общее количество сфер деятельности: {result_4[0]}")

Общее количество сфер деятельности: 294


***

In [90]:
# выводы по предварительному анализу данных
print(f"Общее количество вакансий: {result_1[0]}")
print(f"Общее количество работодателей: {result_2[0]}")
print(f"Общее количество вакансий: {result_3[0]}")
print(f"Общее количество сфер деятельности: {result_4[0]}")

Общее количество вакансий: 49197
Общее количество работодателей: 23501
Общее количество вакансий: 1362
Общее количество сфер деятельности: 294


## 4. Детальный анализ вакансий

1. Напишите запрос, который позволит узнать, сколько (`cnt`) вакансий в каждом регионе (`area`).
Отсортируйте по количеству вакансий в порядке убывания.

In [97]:
# текст запроса
query_4_1 = """
SELECT 
    area.name AS region,
    COUNT(vacancies.*) AS cnt
FROM 
    public.vacancies
LEFT JOIN 
    public.areas AS area ON vacancies.area_id = area.id
GROUP BY 
    area.name
ORDER BY 
    cnt DESC
LIMIT 5;
"""


In [98]:
# результат запроса
cur.execute(query_4_1)
result_4_1 = cur.fetchall()


In [99]:
# Вывод результата
print("Топ-5 регионов по количеству вакансий:")
for i, row in enumerate(result_4_1, 1):
    # row - это кортеж с результатами, индексы: 0 - region, 1 - cnt
    region = row[0]
    cnt = row[1]
    print(f"{i}. {region}: {cnt} вакансий")

Топ-5 регионов по количеству вакансий:
1. Москва: 5333 вакансий
2. Санкт-Петербург: 2851 вакансий
3. Минск: 2112 вакансий
4. Новосибирск: 2006 вакансий
5. Алматы: 1892 вакансий


2. Напишите запрос, чтобы определить у какого количества вакансий заполнено хотя бы одно из двух полей с зарплатой.

In [100]:
# текст запроса
query_4_2 = """
SELECT COUNT(*) AS vacancies_with_salary
FROM public.vacancies
WHERE salary_from IS NOT NULL 
   OR salary_to IS NOT NULL;

"""


In [103]:
# результат запроса

cur.execute(query_4_2)
result_4_2 = cur.fetchall()
# Вывод результата
print(f"Общее количество сфер деятельности: {result_4_2}")

Общее количество сфер деятельности: [(24073,)]


3. Найдите средние значения для нижней и верхней границы зарплатной вилки. Округлите значения до **целого числа**.

In [104]:
# текст запроса
query_4_3 = """
SELECT 
    ROUND(AVG(salary_from)) AS avg_salary_from,
    ROUND(AVG(salary_to)) AS avg_salary_to
FROM public.vacancies;
"""

In [106]:
# результат запроса
cur.execute(query_4_3)
result_4_3 = cur.fetchall()
# Вывод результата
print(f"Средние значения для нижней и верхней границы зарплатной вилки: {result_4_3}")

Средние значения для нижней и верхней границы зарплатной вилки: [(Decimal('71065'), Decimal('110537'))]


4. Напишите запрос, который выведет количество вакансий для каждого сочетания типа рабочего графика (`schedule`) и типа трудоустройства (`employment`), используемого в вакансиях. Результат отсортируйте по убыванию количества.


In [108]:
# текст запроса
query_4_4 = """
SELECT 
    schedule,
    employment,
    COUNT(*) AS vacancies_count
FROM public.vacancies
WHERE schedule IS NOT NULL 
    AND employment IS NOT NULL
GROUP BY schedule, employment
ORDER BY vacancies_count DESC;
"""

In [109]:
# результат запроса
cur.execute(query_4_4)
result_4_4 = cur.fetchall()
# Вывод результата
print(f"Количество вакансий для каждого сочетания типа рабочего графика: {result_4_4}")

Средние значения для нижней и верхней границы зарплатной вилки: [('Полный день', 'Полная занятость', 35367), ('Удаленная работа', 'Полная занятость', 7802), ('Гибкий график', 'Полная занятость', 1593), ('Удаленная работа', 'Частичная занятость', 1312), ('Сменный график', 'Полная занятость', 940), ('Полный день', 'Стажировка', 569), ('Вахтовый метод', 'Полная занятость', 367), ('Полный день', 'Частичная занятость', 347), ('Гибкий график', 'Частичная занятость', 312), ('Полный день', 'Проектная работа', 141), ('Удаленная работа', 'Проектная работа', 133), ('Гибкий график', 'Стажировка', 116), ('Сменный график', 'Частичная занятость', 101), ('Удаленная работа', 'Стажировка', 64), ('Гибкий график', 'Проектная работа', 18), ('Сменный график', 'Стажировка', 12), ('Вахтовый метод', 'Проектная работа', 2), ('Сменный график', 'Проектная работа', 1)]


5. Напишите запрос, выводящий значения поля «Требуемый опыт работы» (`experience`) в порядке возрастания количества вакансий, в которых указан данный вариант опыта.

In [111]:
# текст запроса
query_4_5 = """
SELECT 
    experience,
    COUNT(*) AS vacancies_count
FROM public.vacancies
WHERE experience IS NOT NULL
GROUP BY experience
ORDER BY vacancies_count ASC;
"""

In [112]:
# результат запроса
cur.execute(query_4_5)
result_4_5 = cur.fetchall()
# Вывод результата
print(f"Поля «Требуемый опыт работы: {result_4_5}")

Поля «Требуемый опыт работы: [('Более 6 лет', 1337), ('Нет опыта', 7197), ('От 3 до 6 лет', 14511), ('От 1 года до 3 лет', 26152)]


***

In [114]:
# выводы по детальному анализу вакансий
print(f"{i}. {region}: {cnt} вакансий")

print(f"Общее количество сфер деятельности: {result_4_2}")

print(f"Средние значения для нижней и верхней границы зарплатной вилки: {result_4_3}")

print(f"Количество вакансий для каждого сочетания типа рабочего графика: {result_4_4}")

print(f"Поля «Требуемый опыт работы: {result_4_5}")

5. Алматы: 1892 вакансий
Общее количество сфер деятельности: [(24073,)]
Средние значения для нижней и верхней границы зарплатной вилки: [(Decimal('71065'), Decimal('110537'))]
Количество вакансий для каждого сочетания типа рабочего графика: [('Полный день', 'Полная занятость', 35367), ('Удаленная работа', 'Полная занятость', 7802), ('Гибкий график', 'Полная занятость', 1593), ('Удаленная работа', 'Частичная занятость', 1312), ('Сменный график', 'Полная занятость', 940), ('Полный день', 'Стажировка', 569), ('Вахтовый метод', 'Полная занятость', 367), ('Полный день', 'Частичная занятость', 347), ('Гибкий график', 'Частичная занятость', 312), ('Полный день', 'Проектная работа', 141), ('Удаленная работа', 'Проектная работа', 133), ('Гибкий график', 'Стажировка', 116), ('Сменный график', 'Частичная занятость', 101), ('Удаленная работа', 'Стажировка', 64), ('Гибкий график', 'Проектная работа', 18), ('Сменный график', 'Стажировка', 12), ('Вахтовый метод', 'Проектная работа', 2), ('Сменный гра

1. Географическое распределение вакансий: Анализ показывает, что Алматы является крупнейшим рынком труда среди рассмотренных регионов, с 1892 вакансиями. Это указывает на его ключевую роль как экономического и делового центра.

2. Разнообразие сфер деятельности: В базе данных представлены вакансии из 24 073 различных сфер деятельности. Это свидетельствует о широком разнообразии рынка труда и наличии возможностей для специалистов самых разных профилей.

3. Уровень заработных плат:
    – Средняя минимальная предлагаемая зарплата (нижняя граница вилки) составляет примерно 71 065 единиц (тенге/рублей и т.д.).
    – Средняя максимальная предлагаемая зарплата (верхняя граница вилки) составляет примерно 110 537 единиц.
    – Разница между границами вилки (~39 472 единицы) указывает на то, что итоговый оклад часто является предметом переговоров и зависит от опыта и навыков кандидата.

4. Преобладающий формат занятости: Абсолютно доминирующим форматом является полная занятость с полным рабочим днем (35 367 вакансий). Это стандартный режим работы для большинства отраслей.

5. Популярность удаленного формата: Удаленная работа на условиях полной занятости занимает уверенное второе место (7 802 вакансии), что отражает устойчивый тренд на цифровизацию и гибкие формы занятости.

6. Требования к опыту кандидатов:

   
    – Наибольшее количество вакансий (26 152) ориентировано на специалистов с опытом работы от 1 года до 3 лет. Это самый востребованный сегмент рынка.

   
    – Значительная доля вакансий (14 511) также предназначена для опытных специалистов от 3 до 6 лет.

   
    – Вакансий для кандидатов без опыта (7 197) существенно меньше, чем для опытных, что может указывать на высокую конкуренцию среди начинающих специалистов.

   
    – Меньше всего вакансий для экспертов с опытом более 6 лет (1 337), что может быть связано с узкой специализацией или более высоким уровнем позиций.

**Общий вывод:** Рынок труда, представленный в анализе, является зрелым и разнообразным, с четким центром в Алматы. Он ориентирован в первую очередь на специалистов с опытом работы от 1 до 6 лет, предлагая в основном традиционную полную занятость, но с заметной и устойчивой долей удаленных вакансий. Зарплатные ожидания работодателей имеют значительный диапазон для обсуждения.

## 5. Анализ работодателей

1. Напишите запрос, который позволит узнать, какие работодатели находятся на первом и пятом месте по количеству вакансий.

In [138]:
# текст запроса
query_5_1 = """
WITH employer_ranking AS (
    SELECT 
        e.id AS employer_id,
        e.name AS employer_name,
        COUNT(v.id) AS vacancies_count,
        ROW_NUMBER() OVER (ORDER BY COUNT(v.id) DESC) AS rank
    FROM public.employers e
    LEFT JOIN public.vacancies v ON e.id = v.employer_id
    GROUP BY e.id, e.name
)
SELECT 
    employer_id,
    employer_name,
    vacancies_count,
    rank
FROM employer_ranking
WHERE rank IN (1, 5)
ORDER BY rank;
"""

In [139]:
# результат запроса
cur.execute(query_5_1)
result_5_1 = cur.fetchall()
# Вывод результата
print(f"Какие работодатели находятся на первом и пятом месте по количеству вакансий: {result_5_1}")

Какие работодатели находятся на первом и пятом месте по количеству вакансий: [(1740, 'Яндекс', 1933, 1), (39305, 'Газпром нефть', 331, 5)]


2. Напишите запрос, который для каждого региона выведет количество работодателей и вакансий в нём.
Среди регионов, в которых нет вакансий, найдите тот, в котором наибольшее количество работодателей.


In [89]:
# текст запроса
query_5_2 = """
WITH region_stats AS (
    SELECT 
        a.id AS region_id,
        a.name AS region_name,
        COUNT(DISTINCT e.id) AS employers_count,
        COUNT(DISTINCT v.id) AS vacancies_count
    FROM areas a
    LEFT JOIN employers e ON a.id = e.area
    LEFT JOIN vacancies v ON a.id = v.area_id
    GROUP BY a.id, a.name
),
regions_without_vacancies AS (
    SELECT 
        region_id,
        region_name,
        employers_count
    FROM region_stats
    WHERE vacancies_count = 0
)
SELECT 
    region_id,
    region_name,
    employers_count
FROM regions_without_vacancies
WHERE employers_count = (SELECT MAX(employers_count) FROM regions_without_vacancies)
ORDER BY employers_count DESC;
"""

In [ ]:
# результат запроса
cur.execute(query_5_2)
result_5_2 = cur.fetchall()
# Вывод результата
print(f"Вакансии по регионам: {result_5_2}")

3. Для каждого работодателя посчитайте количество регионов, в которых он публикует свои вакансии. Отсортируйте результат по убыванию количества.


In [10]:
# текст запроса
query_5_3 = """
SELECT 
    e.id AS employer_id,
    e.name AS employer_name,
    COUNT(DISTINCT v.area_id) AS regions_count
FROM 
    employers e
    JOIN vacancies v ON e.id = v.employer_id
GROUP BY 
    e.id, e.name
ORDER BY 
    regions_count DESC,
    e.name;
"""

In [11]:
# результат запроса
cur.execute(query_5_3)
result_5_3 = cur.fetchall()
# Вывод результата
print(f"Вакансии по регионам: {result_5_3}")

Вакансии по регионам: [(1740, 'Яндекс', 181), (2748, 'Ростелеком', 152), (5724811, 'Спецремонт', 116), (5130287, 'Поляков Денис Иванович', 88), (3682876, 'ООО ЕФИН ', 71), (7944, 'Совкомбанк', 63), (3776, 'МТС', 55), (53797, 'ЭФКО, Управляющая компания', 49), (3776815, 'КРОН', 48), (4352, 'Почта России', 48), (622121, 'MCORE', 46), (197135, 'ИК СИБИНТЕК', 46), (1473866, 'Сбербанк-Сервис', 45), (1947314, 'ANCOR', 44), (3177, 'Первый Бит', 43), (78638, 'Тинькофф', 43), (213349, 'АЛНАС', 41), (139, 'IBS', 36), (2180, 'Ozon', 36), (84585, 'Авито', 35), (1237073, 'АТМ АЛЬЯНС', 34), (4872, 'Т1 Интеграция', 34), (44272, 'ЭР-Телеком', 33), (733, 'ЛАНИТ', 29), (172, '1C-Рарус', 28), (5382804, 'Volna.tech', 27), (1035394, 'Красное & Белое, розничная сеть', 27), (6189, 'Bell Integrator', 26), (5390761, 'Совкомбанк Технологии', 26), (114448, 'Playrix', 25), (2381, 'Softline', 25), (49357, 'МАГНИТ, Розничная сеть', 25), (2343, 'Спортмастер', 25), (67611, 'Тензор', 25), (1864933, 'Филиал ФКУ Налог-С

In [14]:
print("Количество регионов для каждого работодателя:")
print("-" * 70)
print(f"{'ID':<8} {'Название работодателя':<40} {'Регионов':<10}")
print("-" * 70)

for row in result_5_3:
    employer_id, employer_name, regions_count = row
    print(f"{employer_id:<8} {employer_name[:38]:<40} {regions_count:<10}")

print("-" * 70)
print(f"Всего работодателей: {len(result_5_3)}")


Количество регионов для каждого работодателя:
----------------------------------------------------------------------
ID       Название работодателя                    Регионов  
----------------------------------------------------------------------
1740     Яндекс                                   181       
2748     Ростелеком                               152       
5724811  Спецремонт                               116       
5130287  Поляков Денис Иванович                   88        
3682876  ООО ЕФИН                                 71        
7944     Совкомбанк                               63        
3776     МТС                                      55        
53797    ЭФКО, Управляющая компания               49        
3776815  КРОН                                     48        
4352     Почта России                             48        
622121   MCORE                                    46        
197135   ИК СИБИНТЕК                              46        
1473866  Сбербанк-С

4. Напишите запрос для подсчёта количества работодателей, у которых не указана сфера деятельности.

In [16]:
# текст запроса
query_5_4 = """
SELECT COUNT(*) AS employers_without_industry
FROM employers e
LEFT JOIN employers_industries ei ON e.id = ei.employer_id
WHERE ei.industry_id IS NULL;
"""

In [17]:
# результат запроса
cur.execute(query_5_4)
result_5_4 = cur.fetchall()
# Вывод результата
print(f"Подсчёт количества работодателей, : {result_5_4}")

Подсчёт количества работодателей, : [(8419,)]


5. Напишите запрос, чтобы узнать название компании, находящейся на третьем месте в алфавитном списке (по названию) компаний, у которых указано четыре сферы деятельности.

In [18]:
# текст запроса
query_5_5 = """
SELECT e.name
FROM employers e
INNER JOIN employers_industries ei ON e.id = ei.employer_id
GROUP BY e.id, e.name
HAVING COUNT(ei.industry_id) = 4
ORDER BY e.name
LIMIT 1 OFFSET 2;
"""

In [19]:
# результат запроса
cur.execute(query_5_5)
result_5_5 = cur.fetchall()
# Вывод результата
print(f"Название компании, находящейся на третьем месте в алфавитном списке: {result_5_5}")

Название компании, находящейся на третьем месте в алфавитном списке: [('2ГИС',)]


6. С помощью запроса выясните, у какого количества работодателей в качестве сферы деятельности указана «Разработка программного обеспечения».


In [20]:
# текст запроса
query_5_6 = """
SELECT COUNT(DISTINCT e.id) AS employer_count
FROM employers e
INNER JOIN employers_industries ei ON e.id = ei.employer_id
INNER JOIN industries i ON ei.industry_id = i.id
WHERE i.name = 'Разработка программного обеспечения';
"""

In [21]:
# результат запроса
cur.execute(query_5_6)
result_5_6 = cur.fetchall()
# Вывод результата
print(f"У какого количества работодателей в качестве сферы деятельности указана «Разработка программного обеспечения»: {result_5_6}")

У какого количества работодателей в качестве сферы деятельности указана «Разработка программного обеспечения»: [(3553,)]


7. Для компании «Яндекс» выведите список [городов-миллионников](https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8), в которых представлены вакансии компании, вместе с количеством вакансий в этих регионах. Также добавьте строку "Total" с общим количеством вакансий компании. Результат отсортируйте по возрастанию количества.

    Если возникнут трудности с этим заданием, посмотрите материалы модуля 6.4 «Как получать данные из веб-источников и API».

In [40]:
# код для получения списка городов-милионников
query_5_7 = """
WITH yandex_vacancies AS (
    -- Все вакансии Яндекса с указанием города
    SELECT 
        v.area_id,
        a.name AS city_name
    FROM vacancies v
    INNER JOIN employers e ON v.employer_id = e.id
    INNER JOIN areas a ON v.area_id = a.id
    WHERE e.name = 'Яндекс'
),
million_city_vacancies AS (
    -- Вакансии Яндекса только в городах-миллионниках
    SELECT 
        yv.city_name,
        COUNT(*) AS vacancy_count
    FROM yandex_vacancies yv
    WHERE yv.city_name IN (
        'Москва', 'Санкт-Петербург', 'Новосибирск', 'Екатеринбург', 
        'Казань', 'Нижний Новгород', 'Челябинск', 'Самара', 'Омск', 
        'Ростов-на-Дону', 'Уфа', 'Красноярск', 'Воронеж', 'Пермь', 
        'Волгоград'
    )
    GROUP BY yv.city_name, yv.area_id
),
total_vacancies AS (
    -- Общее количество вакансий Яндекса в городах-миллионниках
    SELECT 
        'Total' AS city_name,
        SUM(vacancy_count) AS vacancy_count
    FROM million_city_vacancies
)
-- Объединяем города и итоговую строку
SELECT 
    city_name,
    vacancy_count
FROM million_city_vacancies

UNION ALL

SELECT 
    city_name,
    vacancy_count
FROM total_vacancies

ORDER BY vacancy_count ASC;
"""

In [41]:
# текст запроса

In [42]:
# результат запроса
cur.execute(query_5_7)
result_5_7 = cur.fetchall()
# Вывод результата
print(f"Для компании «Яндекс» выведите список городов-миллионников, в которых представлены вакансии компании, вместе с количеством вакансий в этих региона: {result_5_7}")

Для компании «Яндекс» выведите список городов-миллионников, в которых представлены вакансии компании, вместе с количеством вакансий в этих региона: [('Омск', Decimal('21')), ('Челябинск', Decimal('22')), ('Красноярск', Decimal('23')), ('Волгоград', Decimal('24')), ('Ростов-на-Дону', Decimal('25')), ('Пермь', Decimal('25')), ('Казань', Decimal('25')), ('Самара', Decimal('26')), ('Уфа', Decimal('26')), ('Воронеж', Decimal('32')), ('Новосибирск', Decimal('35')), ('Нижний Новгород', Decimal('36')), ('Екатеринбург', Decimal('39')), ('Санкт-Петербург', Decimal('42')), ('Москва', Decimal('54')), ('Total', Decimal('455'))]


***

In [ ]:
# выводы по анализу работодателей

## 6. Предметный анализ

1. Сколько вакансий имеет отношение к данным?

    Считаем, что вакансия имеет отношение к данным, если в её названии содержатся слова `'data'` или `'данн'`.

    *Обратите внимание, что названия вакансий могут быть написаны в любом регистре.*


In [43]:
# текст запроса
query_6_1 = """
SELECT 
    COUNT(*) AS data_related_vacancies_count
FROM vacancies
WHERE 
    LOWER(name) LIKE '%data%' 
    OR LOWER(name) LIKE '%данн%';
"""

In [45]:
# результат запроса

cur.execute(query_6_1)
result_6_1 = cur.fetchall()
# Вывод результата
print(f"вакансия имеет отношение к данным, если в её названии содержатся слова 'data' или 'данн': {result_6_1}")

вакансия имеет отношение к данным, если в её названии содержатся слова 'data' или 'данн': [(1771,)]


2. Сколько есть подходящих вакансий для начинающего дата-сайентиста? Будем считать вакансиями для дата-сайентистов такие, в названии которых есть хотя бы одно из следующих сочетаний:
    * 'data scientist'
    * 'data science'
    * 'исследователь данных'
    * 'ML' (здесь не нужно брать вакансии по HTML)
    * 'machine learning'
    * 'машинн%обучен%'

    **В следующих заданиях мы продолжим работать с вакансиями по этому условию.**

    Считаем вакансиями для специалистов уровня Junior следующие:
    + в названии есть слово “junior” **или**
    + требуемый опыт — «Нет опыта» **или**
    + тип трудоустройства — «Стажировка».


In [46]:
# текст запроса
query_6_2 = """
SELECT 
    COUNT(*) AS junior_data_scientist_vacancies_count
FROM vacancies
WHERE 
    -- Условия для дата-сайентиста
    (
        LOWER(name) LIKE '%data scientist%'
        OR LOWER(name) LIKE '%data science%'
        OR LOWER(name) LIKE '%исследователь данных%'
        OR (LOWER(name) LIKE '%ml%' AND LOWER(name) NOT LIKE '%html%')
        OR LOWER(name) LIKE '%machine learning%'
        OR LOWER(name) LIKE '%машинн%обучен%'
    )
    -- Условия для Junior уровня
    AND (
        LOWER(name) LIKE '%junior%'
        OR experience = 'Нет опыта'
        OR employment = 'Стажировка'
    );
"""

In [48]:
# результат запроса
cur.execute(query_6_2)
result_6_2 = cur.fetchall()
# Вывод результата
print(f"Сколько есть подходящих вакансий для начинающего дата-сайентиста: {result_6_2}")

Сколько есть подходящих вакансий для начинающего дата-сайентиста: [(51,)]


3. Сколько есть вакансий для DS, в которых в качестве ключевого навыка указан SQL или Postgres?

    *Критерии для отнесения вакансии к DS указаны в предыдущем задании.*

In [55]:
# текст запроса
query_6_3 = """
SELECT 
    COUNT(*) AS ds_vacancies_with_sql_or_postgres_count
FROM vacancies
WHERE 
    -- Критерии для отнесения вакансии к DS
    (
        LOWER(name) LIKE '%data scientist%'
        OR LOWER(name) LIKE '%data science%'
        OR LOWER(name) LIKE '%исследователь данных%'
        OR (LOWER(name) LIKE '%ml%' AND LOWER(name) NOT LIKE '%html%')
        OR LOWER(name) LIKE '%machine learning%'
        OR LOWER(name) LIKE '%машинн%обучен%'
    )
    -- Ключевой навык: SQL или Postgres (ищем в поле key_skills)
    AND (
        LOWER(key_skills) LIKE '%sql%'
        OR LOWER(key_skills) LIKE '%postgres%'
        OR LOWER(key_skills) LIKE '%postgresql%'
    );
"""

In [56]:
# результат запроса
cur.execute(query_6_3)
result_6_3 = cur.fetchall()
# Вывод результата
print(f"Сколько есть вакансий для DS, в которых в качестве ключевого навыка указан SQL или Postgres: {result_6_3}")

Сколько есть вакансий для DS, в которых в качестве ключевого навыка указан SQL или Postgres: [(229,)]


4. Проверьте, насколько популярен Python в требованиях работодателей к DS. Для этого вычислите количество вакансий, в которых в качестве ключевого навыка указан Python.

    *Это можно сделать помощью запроса, аналогичного предыдущему.*

In [57]:
# текст запроса
query_6_4 = """
SELECT 
    COUNT(*) AS ds_vacancies_with_python_count
FROM vacancies
WHERE 
    -- Критерии для отнесения вакансии к DS (те же, что и раньше)
    (
        LOWER(name) LIKE '%data scientist%'
        OR LOWER(name) LIKE '%data science%'
        OR LOWER(name) LIKE '%исследователь данных%'
        OR (LOWER(name) LIKE '%ml%' AND LOWER(name) NOT LIKE '%html%')
        OR LOWER(name) LIKE '%machine learning%'
        OR LOWER(name) LIKE '%машинн%обучен%'
    )
    -- Ключевой навык: Python (ищем в поле key_skills)
    AND LOWER(key_skills) LIKE '%python%';
"""

In [58]:
# результат запроса
cur.execute(query_6_4)
result_6_4 = cur.fetchall()
# Вывод результата
print(f"насколько популярен Python в требованиях работодателей к DS: {result_6_4}")

насколько популярен Python в требованиях работодателей к DS: [(357,)]


5. Сколько ключевых навыков в среднем указывают в вакансиях для DS?
Ответ округлите до **двух знаков после точки-разделителя**.

In [77]:
# текст запроса
query_6_5 = """
SELECT 
    ROUND(AVG(skill_count), 2) AS avg_skills_per_ds_vacancy
FROM (
    SELECT 
        v.id,
        CASE 
            WHEN v.key_skills IS NULL OR TRIM(v.key_skills) = '' THEN 0
            ELSE cardinality(
                array_remove(
                    string_to_array(
                        REPLACE(v.key_skills, '\r\n', '\n'), 
                        '\n'
                    ), 
                    ''
                )
            )
        END AS skill_count
    FROM vacancies v
    WHERE 
        (
            LOWER(v.name) LIKE '%data scientist%'
            OR LOWER(v.name) LIKE '%data science%'
            OR LOWER(v.name) LIKE '%исследователь данных%'
            OR (LOWER(v.name) LIKE '%ml%' AND LOWER(v.name) NOT LIKE '%html%')
            OR LOWER(v.name) LIKE '%machine learning%'
            OR LOWER(v.name) LIKE '%машинн%обучен%'
        )
) AS ds_skills;            
"""

In [78]:
# результат запроса
cur.execute(query_6_5)
result_6_5 = cur.fetchall()

# Извлекаем значение из результата (result_6_5 вернет список кортежей)
avg_skills = result_6_5[0][0] if result_6_5 else 0

# Вывод результата
print(f"Сколько ключевых навыков в среднем указывают в вакансиях для DS: {avg_skills:.2f}")

Сколько ключевых навыков в среднем указывают в вакансиях для DS: 0.90


6. Напишите запрос, позволяющий вычислить, какую зарплату для DS в среднем указывают для каждого типа требуемого опыта (уникальное значение из поля `experience`).

    При решении задачи примите во внимание следующее:
    1. Рассматриваем только вакансии, у которых заполнено хотя бы одно из двух полей с зарплатой.
    2. Если заполнены оба поля с зарплатой, то считаем зарплату по каждой вакансии как сумму двух полей, делённую на 2. Если заполнено только одно из полей, то его и считаем зарплатой по вакансии.
    3. Если в расчётах участвует `null`, в результате он тоже даст `null` (посмотрите, что возвращает запрос `select 1 + null`). Чтобы избежать этой ситуацию, мы воспользуемся функцией [coalesce](https://postgrespro.ru/docs/postgresql/9.5/functions-conditional#functions-coalesce-nvl-ifnull), которая заменит `null` на значение, которое мы передадим. Например, посмотрите, что возвращает запрос `select 1 + coalesce(null, 0)`

    Выясните, на какую зарплату в среднем может рассчитывать дата-сайентист с опытом работы от 3 до 6 лет. Результат округлите до **целого числа**.

In [87]:
# текст запроса
query_6_6 = """
WITH ds_vacancies AS (
    SELECT 
        id,
        experience,
        -- Правильный расчет средней зарплаты
        CASE 
            WHEN salary_from IS NOT NULL AND salary_to IS NOT NULL 
                THEN (salary_from + salary_to) / 2.0
            WHEN salary_from IS NOT NULL 
                THEN salary_from
            WHEN salary_to IS NOT NULL 
                THEN salary_to
        END AS avg_salary
    FROM vacancies
    WHERE 
        -- Фильтр на Data Scientist вакансии
        (
            LOWER(name) LIKE '%data scientist%'
            OR LOWER(name) LIKE '%data science%'
            OR LOWER(name) LIKE '%исследователь данных%'
            OR (LOWER(name) LIKE '%ml%' AND LOWER(name) NOT LIKE '%html%')
            OR LOWER(name) LIKE '%machine learning%'
            OR LOWER(name) LIKE '%машинн%обучен%'
        )
        -- Только вакансии с заполненной хотя бы одной границей зарплаты
        AND (salary_from IS NOT NULL OR salary_to IS NOT NULL)
        -- Конкретный опыт работы (проверьте точное название в вашей БД)
        AND experience IN ('От 3 до 6 лет', 'от 3 до 6 лет', 'От 3 до 6')
)
SELECT 
    ROUND(AVG(avg_salary), 0) AS avg_salary_3_to_6_years
FROM ds_vacancies
WHERE avg_salary IS NOT NULL;
"""

In [88]:
# результат запроса
cur.execute(query_6_6)
result_6_6 = cur.fetchall()

# Вывод результата (ИСПРАВЛЕНО сообщение)
print(f"Средняя зарплата для Data Scientist с опытом от 3 до 6 лет: {result_6_6}")

Средняя зарплата для Data Scientist с опытом от 3 до 6 лет: [(Decimal('256454'),)]


***

# Вывод по результатам:

• Вакансий, которые можно отнести к «данным» по названию (содержат data или данн), найдено 1771 — направление достаточно широко представлено на рынке.

• Подходящих вакансий для начинающего дата-сайентиста — 51. Это очень небольшая доля от всех «data»-вакансий (примерно 51 / 1771 ≈ 2.9%), то есть джун-позиций в DS заметно меньше, чем вакансий, связанных с данными в целом.

• Среди DS-вакансий часто требуют навыки работы с базами: SQL/Postgres указаны как ключевой навык в 229 вакансиях — это один из базовых must-have навыков.

• Python встречается в требованиях к DS ещё чаще: 357 упоминаний. По вашим данным Python популярнее, чем SQL/Postgres (357 против 229), что подтверждает его роль как основного языка для DS.

• В среднем в DS-вакансиях указано 0.90 ключевого навыка. Это выглядит заниженным (обычно навыков перечисляют больше), поэтому вероятны ограничения парсинга/заполнения поля «ключевые навыки» (например, не все вакансии содержат структурированный список навыков или навыки извлекались не полностью).

• Средняя зарплата Data Scientist с опытом 3–6 лет — 256 454 (скорее всего, руб.). Это отражает высокий уровень компенсации на мидл-уровне и потенциально сильный рост по сравнению с началом карьеры.

## Общий вывод по проекту

In [ ]:
# подведем итог исследования, обобщите выводы
# здесь можно (это будет плюсом) провести дополнительные исследования данных, сделать прогнозы, продумать варианты продолжения исследования

**Общий вывод:**

исследованная база вакансий описывает зрелый и достаточно диверсифицированный рынок труда с ярко выраженным географическим центром в Алматы (наибольшее число вакансий), при этом основная масса предложений приходится на стандартную полную занятость с полным рабочим днём, а удалённый формат уверенно закрепился как второй по распространённости. 

По опыту рынок сильнее всего ориентирован на кандидатов уровня 1–3 и 3–6 лет, тогда как вакансий без опыта заметно меньше — это повышает конкуренцию среди начинающих и объясняет низкую долю «джун» позиций в узких направлениях.


Направление, связанное с данными, представлено широко: 

по названию найдено 1771 «data/данн»-вакансия, однако непосредственно позиций для начинающего Data Scientist крайне мало (51, около 2.9%), то есть вход в DS сложнее, чем вход в «данные» в целом (аналитика, BI, data engineering и смежные роли). 

В требованиях к DS доминируют технические навыки: 

Python упоминается чаще всего (357), SQL/Postgres также является одним из базовых must-have (229), что отражает практическую направленность вакансий на работу с кодом и данными. 

При этом среднее число ключевых навыков в DS-вакансиях (0.90) выглядит заниженным и, вероятно, связано с особенностями заполнения/парсинга поля навыков, поэтому выводы по «набору навыков» стоит трактовать осторожно.

По зарплатам наблюдается заметный разброс вилок, что указывает на переговорный характер итогового оффера и зависимость от компетенций. 

При этом для Data Scientist на уровне 3–6 лет средняя зарплата высокая (256 454), что подтверждает высокий потенциал роста в профессии после набора опыта. 

В целом рынок предлагает много возможностей, но наиболее конкурентным остаётся старт карьеры (особенно в DS), поэтому для начинающих ключевыми факторами становятся практические навыки (Python/SQL), портфолио/проекты и готовность начинать со смежных «data»-ролей.